# BoM explosion

The BoM table only links each material to its components, month by month. Here every
FIN material gets its whole production chain: the FIN in its own columns and every
material → component pair of the chain under it, with quantities for the year.
By plant, FIN material and year.

In [1]:
from pathlib import Path

import pandas as pd

DATA = Path("data/bom.csv")  # the .xlsx works too
OUTPUT = Path("output/bom_explosion.csv")

## 1. Loading

Material numbers and types are codes, so they're read as text: no `8002.0`, no lost
leading zeros. The CSV export writes quantities with thousands separators, like
`"1,726.00"`, so the commas go before converting.

In [2]:
CODES = [
    "plant_id",
    "produced_material",
    "produced_material_production_type",
    "produced_material_release_type",
    "component_material",
    "component_material_production_type",
    "component_material_release_type",
]
QUANTITIES = ["produced_material_quantity", "component_material_quantity"]


def load_bom(path: Path) -> pd.DataFrame:
    """Reads the BoM from .csv or .xlsx. Empty codes become NaN."""
    read = pd.read_excel if path.suffix.lower() == ".xlsx" else pd.read_csv
    bom = read(path, dtype=str)
    bom.columns = bom.columns.str.strip().str.lower()
    missing = {"year", "month", *CODES, *QUANTITIES} - set(bom.columns)
    if missing:
        raise ValueError(f"{path} has no column(s) {sorted(missing)}")
    for column in CODES:
        codes = bom[column].str.strip()
        bom[column] = codes.where(codes != "")
    for column in QUANTITIES:
        bom[column] = bom[column].str.replace(",", "").astype(float)
    bom[["year", "month"]] = bom[["year", "month"]].astype(int)
    return bom


bom = load_bom(DATA)
bom.info()

<class 'pandas.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 11 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                1320 non-null   int64  
 1   month                               1320 non-null   int64  
 2   produced_material                   1320 non-null   str    
 3   produced_material_production_type   1320 non-null   str    
 4   produced_material_release_type      1320 non-null   str    
 5   produced_material_quantity          1320 non-null   float64
 6   component_material                  1320 non-null   str    
 7   component_material_production_type  480 non-null    str    
 8   component_material_release_type     1320 non-null   str    
 9   component_material_quantity         1320 non-null   float64
 10  plant_id                            1320 non-null   str    
dtypes: float64(2), int64(2), str(7)
memory usage: 113.6 KB

## 2. What's in the file

Each row is one material → component pair in one month. The first 11 rows are a whole
chain: FIN material 10000 in January.

In [3]:
bom.head(11)

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.0,50000,8002,PROD,990.0,RLT_10
1,2024,1,50000,8002,PROD,859.0,80070,8007,PROD,879.0,RLT_10
2,2024,1,50000,8002,PROD,859.0,90000,NaN,ADD,50.0,RLT_10
3,2024,1,50000,8002,PROD,859.0,90001,NaN,ADD,20.0,RLT_10
4,2024,1,80070,8007,PROD,929.0,80010,8001,PROD,3626.0,RLT_10
5,2024,1,80070,8007,PROD,929.0,90002,NaN,ADD,30.0,RLT_10
6,2024,1,80070,8007,PROD,929.0,90003,NaN,ADD,11.0,RLT_10
7,2024,1,80010,8001,PROD,1726.0,80000,8000,PROD,1818.0,RLT_10
8,2024,1,80010,8001,PROD,1726.0,90004,NaN,ADD,101.0,RLT_10
9,2024,1,80000,8000,PROD,1980.0,70000,NaN,RM,2668.0,RLT_10


In [4]:
bom.groupby(["year", "plant_id"]).agg(
    rows=("month", "size"), months=("month", "nunique")
)

rows  months
year plant_id              
2024 RLT_10     264      12
     RLT_14     792      12
     RLT_16     264      12

Only 2024, all 12 months, three plants. The task asks for 2024 and 2025, but there's
no 2025 in the file. Nothing below depends on that: every year in the data gets its
own explosion.

In [5]:
pd.crosstab(
    bom["produced_material_release_type"], bom["produced_material_production_type"]
)

produced_material_production_type,8000,8001,8002,8007
produced_material_release_type,,,,
FIN,0,0,120,0
PROD,240,240,360,360


In [6]:
pd.crosstab(
    bom["component_material_release_type"],
    bom["component_material_production_type"].fillna("empty"),
)

component_material_production_type,8000,8001,8002,8007,empty
component_material_release_type,,,,,
ADD,0,0,0,0,720
PROD,120,120,120,120,0
RM,0,0,0,0,120


Only FIN and PROD materials have components. ADD and RM come up only as components and
have no production type. The stages are 8000 polysilicon → 8001 ingot → 8007 lapped
wafer → 8002 polished wafer, and every FIN material is a polished wafer.

In [7]:
january_50000 = (
    (bom["plant_id"] == "RLT_10")
    & (bom["month"] == 1)
    & (bom["produced_material"] == "50000")
)
bom.loc[
    january_50000,
    [
        "produced_material",
        "produced_material_quantity",
        "component_material",
        "component_material_quantity",
    ],
]

,produced_material,produced_material_quantity,component_material,component_material_quantity
1,50000,859.0,80070,879.0
2,50000,859.0,90000,50.0
3,50000,859.0,90001,20.0


A material's produced quantity is repeated on each of its component rows: 859 here,
three times. For the year it's taken once per month and then summed. Component
quantities are summed as they are.

In [8]:
months_per_pair = bom.groupby(
    ["plant_id", "year", "produced_material", "component_material"]
)["month"].nunique()
months_per_pair.value_counts()

month
12    110
Name: count, dtype: int64

All 110 pairs are there in all 12 months, so the structure doesn't change during the
year. If it did, the year's explosion would simply have both the old and the new pairs.

In [9]:
produced_in = bom.groupby("produced_material")["plant_id"].nunique()
print("produced materials in more than one plant:", produced_in.gt(1).sum())

used_in = bom.drop_duplicates(["plant_id", "component_material"])
in_several = used_in[used_in.duplicated("component_material", keep=False)]
in_several.groupby(["component_material", "component_material_release_type"])[
    "plant_id"
].agg(lambda plants: ", ".join(sorted(plants)))

produced materials in more than one plant: 0


component_material  component_material_release_type
90010               ADD                                RLT_10, RLT_14
90035               ADD                                RLT_14, RLT_16
90045               ADD                                RLT_14, RLT_16
Name: plant_id, dtype: str

Produced materials belong to one plant, but three additives are used in more than one.
A BoM belongs to its plant anyway, so a chain is always followed within one plant and
year.

In [10]:
fin = bom[bom["produced_material_release_type"] == "FIN"]
fin.groupby(["plant_id", "produced_material"]).agg(
    production_type=("produced_material_production_type", "first"),
    components=("component_material", "nunique"),
    component=("component_material", "first"),
    component_release_type=("component_material_release_type", "first"),
    component_production_type=("component_material_production_type", "first"),
)

production_type  components component  \
plant_id produced_material                                         
RLT_10   10000                        8002           1     50000   
         10001                        8002           1     50001   
RLT_14   10002                        8002           1     50002   
         10003                        8002           1     50003   
         10004                        8002           1     50004   
         10005                        8002           1     50005   
         10006                        8002           1     50006   
         10009                        8002           1     50009   
RLT_16   10007                        8002           1     50007   
         10008                        8002           1     50008   

                           component_release_type component_production_type  
plant_id produced_material                                                   
RLT_10   10000                               PROD                      8002  
         10001                               PROD                      8002  
RLT_14   10002                               PROD                      8002  
         10003                               PROD                      8002  
         10004                               PROD                      8002  
         10005                               PROD                      8002  
         10006                               PROD                      8002  
         10009                               PROD                      8002  
RLT_16   10007                               PROD                      8002  
         10008                               PROD                      8002

10 FIN materials, each consisting of exactly one PROD material: the same wafer under its
production number. That PROD material is where each chain starts. As in the task's
example, the FIN → PROD pair itself isn't a row of the report.

Everything the explosion relies on, checked in one place. With a new file, this is the
cell to look at first.

In [11]:
def rows(frame: pd.DataFrame) -> set[tuple]:
    """The rows of a frame as a set, to compare frames regardless of order."""
    return set(frame.itertuples(index=False, name=None))


def check_bom(bom: pd.DataFrame) -> pd.Series:
    """What the explosion relies on. All of it should be True."""
    month = ["plant_id", "year", "month", "produced_material"]
    produced = bom[["plant_id", "year", "produced_material"]]
    is_prod = bom["component_material_release_type"] == "PROD"
    prod_components = bom.loc[is_prod, ["plant_id", "year", "component_material"]]
    is_fin = bom["produced_material_release_type"] == "FIN"
    fin_components = bom.loc[is_fin, "component_material_release_type"]
    columns = ["plant_id", "year", "material", "release_type", "production_type"]
    types = pd.concat(
        bom[
            [
                "plant_id",
                "year",
                f"{side}_material",
                f"{side}_material_release_type",
                f"{side}_material_production_type",
            ]
        ].set_axis(columns, axis=1)
        for side in ("produced", "component")
    ).drop_duplicates()
    checks = {
        "one row per pair and month": (
            not bom.duplicated([*month, "component_material"]).any()
        ),
        "a material's produced quantity is the same on all its rows in a month": (
            bom.groupby(month)["produced_material_quantity"].nunique().eq(1).all()
        ),
        "only FIN and PROD materials have components": (
            bom["produced_material_release_type"].isin(["FIN", "PROD"]).all()
        ),
        "FIN materials consist of PROD materials only": (
            fin_components.eq("PROD").all()
        ),
        "every PROD component has components of its own": (
            rows(prod_components) <= rows(produced)
        ),
        "a material has the same types in all rows of a plant and year": (
            not types.duplicated(["plant_id", "year", "material"]).any()
        ),
    }
    return pd.Series(checks)


bom_checks = check_bom(bom)
assert bom_checks.all(), bom_checks[~bom_checks]
bom_checks

one row per pair and month                                               True
a material's produced quantity is the same on all its rows in a month    True
only FIN and PROD materials have components                              True
FIN materials consist of PROD materials only                             True
every PROD component has components of its own                           True
a material has the same types in all rows of a plant and year            True
dtype: bool

## 3. Explosion

1. `annual_bom` sums the months into a year, per plant and pair.
2. `explode` walks down from each FIN material, a level at a time. Level 1 is the pairs
   of the PROD material the FIN consists of, level 2 the pairs of that material's PROD
   components, and so on until only ADD and RM are left. One merge per level for all
   FIN materials at once, no loop over materials.
3. `bom_explosion` gives it the report's column names (lowercase, as the task asks) and
   order.

The quantities are for the whole year: `fin_production_quantity` and
`prod_material_production_quantity` are what the FIN and the pair's material produced,
`component_consumption_quantity` is how much of the component that material used.
They're the BoM's own figures, nothing is recalculated along the chain.

In [12]:
PAIR = [
    "plant_id",
    "year",
    "produced_material",
    "produced_material_release_type",
    "produced_material_production_type",
    "component_material",
    "component_material_release_type",
    "component_material_production_type",
]


def annual_bom(bom: pd.DataFrame) -> pd.DataFrame:
    """One row per plant, year and pair: the material's production and the
    component's consumption over the year."""
    material = ["plant_id", "year", "produced_material"]
    # the produced quantity is on every component row, so take it once a month
    production = (
        bom.drop_duplicates([*material, "month"])
        .groupby(material, as_index=False)["produced_material_quantity"]
        .sum()
    )
    consumption = bom.groupby(PAIR, as_index=False, dropna=False)[
        "component_material_quantity"
    ].sum()
    annual = consumption.merge(production, on=material, validate="many_to_one")
    return annual.round(3)  # float noise from the sums

In [13]:
FIN = {
    "produced_material": "fin_material_id",
    "produced_material_release_type": "fin_material_release_type",
    "produced_material_production_type": "fin_material_production_type",
    "produced_material_quantity": "fin_production_quantity",
}


def explode(annual: pd.DataFrame, max_levels: int = 20) -> pd.DataFrame:
    """Every pair in the chain of every FIN material, with the FIN's columns and
    the pair's level in the chain."""
    carried = ["plant_id", "year", *FIN.values()]
    fin = annual[annual["produced_material_release_type"] == "FIN"].rename(columns=FIN)
    # materials to expand next, each with the FIN it belongs to
    todo = fin[[*carried, "component_material"]]
    levels = []
    for level in range(1, max_levels + 1):
        todo = todo.rename(columns={"component_material": "produced_material"})
        found = todo.merge(annual, on=["plant_id", "year", "produced_material"])
        found["level"] = level
        if found.empty:
            return pd.concat(levels, ignore_index=True) if levels else found
        levels.append(found)
        todo = found[[*carried, "component_material"]]
    raise ValueError(f"a chain is deeper than {max_levels} levels, is there a cycle?")

In [14]:
REPORT = {
    "plant_id": "plant",
    "produced_material": "prod_material_id",
    "produced_material_release_type": "prod_material_release_type",
    "produced_material_production_type": "prod_material_production_type",
    "produced_material_quantity": "prod_material_production_quantity",
    "component_material": "component_id",
    "component_material_quantity": "component_consumption_quantity",
}
COLUMNS = [
    "plant",
    "fin_material_id",
    "fin_material_release_type",
    "fin_material_production_type",
    "fin_production_quantity",
    "prod_material_id",
    "prod_material_release_type",
    "prod_material_production_type",
    "prod_material_production_quantity",
    "component_id",
    "component_material_release_type",
    "component_material_production_type",
    "component_consumption_quantity",
    "year",
]


def bom_explosion(bom: pd.DataFrame) -> pd.DataFrame:
    """The report: each FIN material's chain from the top down, by plant and year."""
    chains = explode(annual_bom(bom)).sort_values(
        [
            "plant_id",
            "fin_material_id",
            "year",
            "level",
            "produced_material",
            "component_material",
        ]
    )
    return chains.rename(columns=REPORT)[COLUMNS].reset_index(drop=True)

## 4. Result

In [15]:
report = bom_explosion(bom)
report.shape

(100, 14)

FIN material 10000 from the top down: its PROD material 50000 (8002), then the lapped
wafer (8007), the ingot (8001) and the polysilicon (8000) with the raw material it's
made of. ADD additives at every stage.

In [16]:
report[report["fin_material_id"] == "10000"]

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007,11303.0,2024
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024
3,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001,41769.0,2024
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,NaN,355.0,2024
5,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,NaN,121.0,2024
6,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000,23360.0,2024
7,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,NaN,1242.0,2024
8,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0,2024
9,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,90005,ADD,NaN,829.0,2024


In [17]:
report.groupby(["plant", "fin_material_id", "year"]).agg(
    rows=("component_id", "size"),
    stages=("prod_material_production_type", lambda types: " → ".join(types.unique())),
)

rows                     stages
plant  fin_material_id year                                 
RLT_10 10000           2024    10  8002 → 8007 → 8001 → 8000
       10001           2024    10  8002 → 8007 → 8001 → 8000
RLT_14 10002           2024    10  8002 → 8007 → 8001 → 8000
       10003           2024    10  8002 → 8007 → 8001 → 8000
       10004           2024    10  8002 → 8007 → 8001 → 8000
       10005           2024    10  8002 → 8007 → 8001 → 8000
       10006           2024    10  8002 → 8007 → 8001 → 8000
       10009           2024    10  8002 → 8007 → 8001 → 8000
RLT_16 10007           2024    10  8002 → 8007 → 8001 → 8000
       10008           2024    10  8002 → 8007 → 8001 → 8000

Every FIN material has the whole chain, 10 rows each. Checks against the source:

In [18]:
def check_report(report: pd.DataFrame, bom: pd.DataFrame) -> pd.Series:
    """The result against the source. All of it should be True."""
    pair = ["plant", "year", "prod_material_id", "component_id"]
    below_fin = bom[bom["produced_material_release_type"] != "FIN"]
    bom_pairs = below_fin[
        ["plant_id", "year", "produced_material", "component_material"]
    ]
    chain = report.groupby(["plant", "fin_material_id", "year"])
    stages = chain["prod_material_production_type"].unique().map(set)
    ends = chain["component_material_release_type"].unique().map(set)
    consumed = report.drop_duplicates(pair)["component_consumption_quantity"].sum()
    checks = {
        "columns as in the task": list(report.columns) == COLUMNS,
        "no pair twice in a chain": (
            not report.duplicated(["fin_material_id", *pair]).any()
        ),
        "every BoM pair below FIN level is in the report, and nothing else": (
            rows(report[pair]) == rows(bom_pairs)
        ),
        "every chain has all four stages": (
            stages.map({"8002", "8007", "8001", "8000"}.issubset).all()
        ),
        "every chain ends in raw material": ends.map(lambda types: "RM" in types).all(),
        "consumption adds up to the BoM's": (
            round(consumed, 3)
            == round(below_fin["component_material_quantity"].sum(), 3)
        ),
    }
    return pd.Series(checks)


report_checks = check_report(report, bom)
assert report_checks.all(), report_checks[~report_checks]
report_checks

columns as in the task                                               True
no pair twice in a chain                                             True
every BoM pair below FIN level is in the report, and nothing else    True
every chain has all four stages                                      True
every chain ends in raw material                                     True
consumption adds up to the BoM's                                     True
dtype: bool

## 5. Saving

In [19]:
OUTPUT.parent.mkdir(exist_ok=True)
report.to_csv(OUTPUT, index=False)
print(f"{len(report)} rows → {OUTPUT}")

100 rows → output/bom_explosion.csv
